In [1]:
#1)VectorStoreRetriever	Vector-based	Embedding similarity search	General-purpose RAG
# ConversationalRetrievalChain

from langchain.chat_models import ChatOpenAI
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import ConversationalRetrievalChain
from langchain.agents import initialize_agent, AgentType
from langchain.tools import StructuredTool
from langchain.memory import ConversationBufferMemory
from langchain.text_splitter import CharacterTextSplitter
import os
from dotenv import load_dotenv

# 2️⃣ Load API keys
load_dotenv(".env")  # Loads from the .env file in the same directory

openai_key = os.getenv("OPENAI_API_KEY")
if not openai_key:
    raise ValueError("OPENAI_API_KEY not found in .env file. Please check your .env file.")
os.environ["OPENAI_API_KEY"] = openai_key


# 1. LLM & Embeddings
llm = ChatOpenAI(temperature=0)

# 2. Vector DB — load and split text
with open("sample.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

# 🧠 Split the text into smaller chunks
splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
texts = splitter.split_text(text_data)

embedding = OpenAIEmbeddings()
vectorstore = FAISS.from_texts(texts, embedding)
retriever = vectorstore.as_retriever()

# 3. Conversational RAG chain
rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    return_source_documents=False
)

# 4. Memory for chat history
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# 5. Wrap RAG as a StructuredTool
def rag_tool_fn(question: str) -> str:
    return rag_chain.invoke({
        "question": question,
        "chat_history": []
    })["answer"]

rag_tool = StructuredTool.from_function(
    name="RAG_QA",
    description="Use this to answer questions about LangChain.",
    func=rag_tool_fn
)

# 6. Create agent (verbose=False for clean output)
agent = initialize_agent(
    tools=[rag_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=False,
    memory=memory,
    handle_parsing_errors=True
)

# 7. Run conversation
print("1️⃣ First question")
res1 = agent.run("What is LangChain?")
print("Answer:", res1)

print("\n2️⃣ Follow-up")
res2 = agent.run("Who created it?")
print("Answer:", res2)

print("\n3️⃣ Ask again")
res3 = agent.run("Explain LangChain again simply.")
print("Answer:", res3)


C:\Users\premchandar\AppData\Local\Temp\ipykernel_7964\4128499504.py:25: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  llm = ChatOpenAI(temperature=0)
C:\Users\premchandar\AppData\Local\Temp\ipykernel_7964\4128499504.py:35: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import OpenAIEmbeddings`.
  embedding = OpenAIEmbeddings()
C:\Users\premchandar\AppData\Local\Temp\ipykernel_7964\4128499504.py:63: LangChainDeprecationWarning: The function `initialize_agent` was 

1️⃣ First question
Answer: LangChain is an open-source framework created by Harrison Chase in October 2022. It simplifies the creation of applications using large language models (LLMs) by enabling developers to build context-aware reasoning applications through connecting language models to other data sources and allowing interaction with the environment. Key features include Chains, which are sequences of calls to components, including other chains. LangChain provides a standard interface for chains, integrations with other tools, and end-to-end chains for common applications. It supports multiple RAG patterns such as VectorStoreRetriever, ConversationalRetrievalChain, and MultiQueryRetriever.

2️⃣ Follow-up
Answer: LangChain was created by Harrison Chase in October 2022.

3️⃣ Ask again
Answer: LangChain is a tool that helps developers create applications that can understand and interact with their environment using language models. It allows for building sequences of actions called 